In [75]:
import re 
import pandas as pd 
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import torch 
from transformers import AutoTokenizer, Trainer, TrainingArguments, \
    BertForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score

In [76]:
df = pd.read_json("../data/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50336 entries, 0 to 50335
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   도메인        50336 non-null  str  
 1   카테고리       50336 non-null  str  
 2   대화셋일련번호    50336 non-null  str  
 3   화자         50336 non-null  str  
 4   문장번호       50336 non-null  int64
 5   고객의도       50336 non-null  str  
 6   상담사의도      50336 non-null  str  
 7   QA         50336 non-null  str  
 8   고객질문(요청)   50336 non-null  str  
 9   상담사질문(요청)  50336 non-null  str  
 10  고객답변       50336 non-null  str  
 11  상담사답변      50336 non-null  str  
 12  개체명        50336 non-null  str  
 13  용어사전       50336 non-null  str  
 14  지식베이스      50336 non-null  str  
dtypes: int64(1), str(14)
memory usage: 13.7 MB


In [77]:
df.rename(
    columns = {
        '도메인' : ' 도메인'
    }, inplace = True
)

In [78]:
df.iloc[18: 30]

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
18,다산콜센터,일반행정 문의,B2240,고객,19,지방세납부,,Q,다른곳에서는 납부할수 없습니까?,,,,,납부방법/지방세/ 위텍스/ 납부,
19,다산콜센터,일반행정 문의,B2240,상담사,20,,지방세납부,A,,,,이용하시는 은행의 사이트에서도 지방세 납부가 가능합니다.,"은행, 사이트",은행/공공기관,"사이트,공공기관"
20,다산콜센터,일반행정 문의,B2241,고객,1,지방세납부,,Q,지방세는 조회할 수 있습니까?,,,,"지방세, 조회",지방세/세금,"조회,세금"
21,다산콜센터,일반행정 문의,B2241,상담사,2,,지방세납부,A,,,,간단한 본인확인 후 안내해드리겠습니다.,"본인확인, 안내",,안내
22,다산콜센터,일반행정 문의,B2241,고객,3,지방세납부,,A,,,알겠습니다.,,,,
23,다산콜센터,일반행정 문의,B2241,상담사,4,,지방세납부,Q,,지금 전화거신 핸드폰이 본인명의 맞습니까?,,,"핸드폰, 본인명의",핸드폰/전자기기,"본인명의,전자기기"
24,다산콜센터,일반행정 문의,B2241,고객,5,지방세납부,,A,,,맞습니다.,,,,
25,다산콜센터,일반행정 문의,B2241,상담사,6,,지방세납부,Q,,생년월일을 말씀해주시겠습니까?,,,생년월일,생년월일/개인정보,"생년월일,개인정보"
26,다산콜센터,일반행정 문의,B2241,고객,7,지방세납부,,A,,,OO월OO일 입니다.,,,,
27,다산콜센터,일반행정 문의,B2241,상담사,8,,지방세납부,Q,,살고계신 주소는 어디입니까?,,,주소,주소/개인정보,"주소,개인정보"


In [79]:
# 데이터프레임 로드 하고 컬럼의 이름들을 확인! 
df.columns.str.strip()

Index(['도메인', '카테고리', '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA',
       '고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변', '개체명', '용어사전', '지식베이스'],
      dtype='str')

In [80]:
df.columns.map( lambda x : x.strip() )

Index(['도메인', '카테고리', '대화셋일련번호', '화자', '문장번호', '고객의도', '상담사의도', 'QA',
       '고객질문(요청)', '상담사질문(요청)', '고객답변', '상담사답변', '개체명', '용어사전', '지식베이스'],
      dtype='str')

In [81]:
df.columns = [ x.strip() for x in df.columns ]

In [82]:
df.head()

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
2,다산콜센터,일반행정 문의,B2240,고객,3,지방세납부,,Q,은행 어플에서도 됩니까?,,,,"은행, 어플",은행/공공기관,"어플,공공기관"
3,다산콜센터,일반행정 문의,B2240,상담사,4,,지방세납부,Q,,어떤 은행을 이용하고 계십니까?,,,은행,은행/공공기관,"은행,공공기관"
4,다산콜센터,일반행정 문의,B2240,고객,5,지방세납부,,A,,,기업은행을 이용하고 있습니다.,,기업은행,기업은행/상호,"기업은행,상호"


In [83]:
# 필요한 컬럼을 제외하고 나머지는 제외 
df2 = df[['고객질문(요청)', '상담사답변']]

In [84]:
df2.rename(
    columns = {
        '고객질문(요청)' : '고객질문'
    }, inplace = True
)

In [85]:
# 텍스트 정규화 
def normalize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", ' ', str(text))
    text = re.sub(r"\s+", ' ', text).strip()

    return text

In [86]:
df2 = df2.map(normalize)

In [87]:
flag1 = (df2['고객질문'] != '') & (df2['상담사답변'].shift(-1) != '') & (df['문장번호'] == 1)
df3 = df2.loc[flag1, ]

In [88]:
# flag1을 한칸씩 밑으로 내리면 상담사의 답변
df3['상담사답변'] = df2.loc[flag1.shift(1).fillna(False), '상담사답변'].tolist()

In [89]:
df3.head()

,고객질문,상담사답변
0,지방세를 내려면 어떻게 해야됩니까,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.
20,지방세는 조회할 수 있습니까,간단한 본인확인 후 안내해드리겠습니다.
40,서울시주최 페스티벌 예매해놨는데 예정대로 진행됩니까,현재로썬 진행될 예정입니다.
60,청년저축계좌 지금 신청할 수 있습니까,죄송하지만 이미 신청기간이 지났습니다.
200,보건증 무인발급기로 출력할 수 있습니까,보건증은 모든 보건소에서 지원하는게 아니라서 검사받은 보건소 확인이 필요합니다.


In [90]:
df3.info()

<class 'pandas.DataFrame'>
Index: 1087 entries, 0 to 50316
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   고객질문    1087 non-null   str  
 1   상담사답변   1087 non-null   str  
dtypes: str(2)
memory usage: 149.2 KB


In [91]:
# 기존의 데이터의 질문과 답변은 정상적인 답변 labels를 1로 채워준다. 
df3['labels'] = 1

In [92]:
# 기존의 질문은 유지한채 답변은 바꿔서 labels가 0인 구간을 생성 
answer_list = [
    ['A', 'a'], 
    ['B', 'b'], 
    ['C', 'c'], 
    ['D', 'd']
]
neg_list = []
for q, a in answer_list:
    # q : 질문
    # a : 답변

    cand = []
    for q2, a2 in answer_list:
        # a2 : 답변들의 목록
        if a != a2:
            cand.append(a2)
    # cand 틀린 답변의 목록에서 무작위로 하나를 선택 
    neg_a = np.random.choice(cand)
    neg_list.append([q, neg_a])

neg_list


[['A', np.str_('d')],
 ['B', np.str_('a')],
 ['C', np.str_('d')],
 ['D', np.str_('c')]]

In [93]:
answer_list2 = df3[['고객질문', '상담사답변']].values.tolist()

In [94]:
neg_list2 = []
for q, a in answer_list2:
    cand = [ a2 for q2, a2 in answer_list2 if a != a2 ]

    neg_a = np.random.choice(cand)
    neg_list2.append( [q, neg_a] )
neg_list2

[['지방세를 내려면 어떻게 해야됩니까', np.str_('예 성실히 답변 드리겠습니다.')],
 ['지방세는 조회할 수 있습니까', np.str_('네 건강보험이 적용됩니다.')],
 ['서울시주최 페스티벌 예매해놨는데 예정대로 진행됩니까', np.str_('네 서울도서관을 운영중입니다.')],
 ['청년저축계좌 지금 신청할 수 있습니까', np.str_('예.거주지 근처 주민센터에서 발급 가능하십니다.')],
 ['보건증 무인발급기로 출력할 수 있습니까',
  np.str_('우리동네키움센터는 초등학생 누구나 돌봄을 필요로 할 때 이용할 수 있는 곳입니다.')],
 ['안녕하세요. 제가 이번에 공무원 시험을 봤는데 시험 결과는 언제 나옵니까', np.str_('네 있습니다.')],
 ['농촌일손돕기 봉사 신청하려고합니다.',
  np.str_('취득세란 일정한 자산을 취득했을 때 이에 대해 부과하는 조세를 뜻하는 용어입니다.')],
 ['주말농장 신청하려고 합니다.', np.str_('청년의 다양한 상황과 필요에 맞게 사용이 가능합니다.')],
 ['안녕하세요. 저 장학생 문의하려 합니다.', np.str_('네 무엇이 궁금하신가요')],
 ['안녕하세요. 유기동물 복지 관련 문의드리려합니다.', np.str_('건강보험관리공단에 신청하시면 됩니다.')],
 ['가산도서관 지금 이용 불가입니까', np.str_('물론 접수 가능합니다.')],
 ['농기계 대여 하려합니다.', np.str_('네 합니다')],
 ['서울시민체육센터 지금 운영합니까', np.str_('마라톤처럼 코스를 정해두고 책 읽기 경주하는겁니다.')],
 ['기초수급자 신청하려고 합니다.',
  np.str_('자동차세는 보통 1월에서 6월까지의 자동차세를 6월에 후불 납부하고 7월에서 12월까지의 자동차세를 12월에 후불 납부하고 있습니다.')],
 ['운전면허 갱신하려합니다.', np.str_('결과는 9월 20일로 예정되어있어요.')],
 ['이번에 한국사 시험 미뤄집니까', np

In [95]:
neg_df = pd.DataFrame(neg_list2, columns = ['고객질문', '상담사답변'])
neg_df['labels'] = 0
neg_df.head()

,고객질문,상담사답변,labels
0,지방세를 내려면 어떻게 해야됩니까,예 성실히 답변 드리겠습니다.,0
1,지방세는 조회할 수 있습니까,네 건강보험이 적용됩니다.,0
2,서울시주최 페스티벌 예매해놨는데 예정대로 진행됩니까,네 서울도서관을 운영중입니다.,0
3,청년저축계좌 지금 신청할 수 있습니까,예.거주지 근처 주민센터에서 발급 가능하십니다.,0
4,보건증 무인발급기로 출력할 수 있습니까,우리동네키움센터는 초등학생 누구나 돌봄을 필요로 할 때 이용할 수 있는 곳입니다.,0


In [96]:
# df3와 neg_df을 단순 행 결합 
dataset_df = pd.concat(
    [df3, neg_df], axis = 0, ignore_index=True
)

In [97]:
dataset_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2174 entries, 0 to 2173
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   고객질문    2174 non-null   str  
 1   상담사답변   2174 non-null   str  
 2   labels  2174 non-null   int64
dtypes: int64(1), str(2)
memory usage: 297.1 KB


In [98]:
dataset_df['labels'].value_counts()

labels
1    1087
0    1087
Name: count, dtype: int64

In [99]:
train_df, test_df = train_test_split(
    dataset_df, test_size = 0.5, random_state=42, stratify= dataset_df['labels']
)

In [100]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))
# DatasetDict
ds = DatasetDict(
    {
        'train' : train_ds, 
        'validation' : test_ds
    }
)

In [101]:
ds

DatasetDict({
    train: Dataset({
        features: ['고객질문', '상담사답변', 'labels'],
        num_rows: 1087
    })
    validation: Dataset({
        features: ['고객질문', '상담사답변', 'labels'],
        num_rows: 1087
    })
})

In [102]:
MODEL_NAME = 'beomi/kcbert-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast = False)
max_len = 128

def token_fn(batch):
    # batch -> dict{'고객질문', '상담사답변'}
    tok = tokenizer(
        batch['고객질문'], 
        batch['상담사답변'], 
        truncation = True, 
        max_length = max_len
    )

    # 토큰화된 데이터에서 token_type_ids는 kobert 모델에서는 사용하지 않는다. 
    # 해당 키를 제거 
    tok.pop("token_type_ids", None)
    return tok

# remove_columns 매개변수 -> 토큰화를 하고 제외시킬 컬럼을 지정 
tok_ds = ds.map(
    token_fn, 
    batched = True, 
    remove_columns = [col for col in dataset_df.columns if col not in ['labels']], 
    # 캐시 사용 안함 
    load_from_cache_file = False
)

Map: 100%|██████████| 1087/1087 [00:00<00:00, 47586.43 examples/s]


In [103]:
tok_ds['train'][0]['input_ids']

[2,
 29896,
 599,
 4307,
 8431,
 9554,
 8518,
 4040,
 17,
 3,
 8270,
 9656,
 24,
 4118,
 28169,
 17024,
 9569,
 4230,
 1410,
 4334,
 4984,
 4715,
 8102,
 16533,
 10009,
 3]

In [104]:
# 배치마다 동적으로 padding 토큰을 추가 
collator = DataCollatorWithPadding(
    tokenizer= tokenizer
)

In [105]:
# 학습 모델 정의 
# BertModel -> dropout -> Linear
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels = 2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7628.45it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [106]:
# 평가 함수 
def metrics(eval_pred):
    logits, y = eval_pred
    pred = np.argmax(logits, axis = -1)
    return {
        'accuracy_score' : accuracy_score(pred, y), 
        'f1_score' : f1_score(pred, y)
    }

In [107]:
args = TrainingArguments(
    output_dir= './model2', 
    eval_strategy='epoch', 
    save_strategy='epoch', 
    learning_rate=5e-05, 
    weight_decay=0.01, 
    warmup_steps=0.1, 
    num_train_epochs=3, 
    load_best_model_at_end=True, 
    metric_for_best_model='f1_score', 
    greater_is_better=True
)

In [108]:
trainer = Trainer(
    model = model, 
    args = args, 
    train_dataset= tok_ds['train'], 
    eval_dataset= tok_ds['validation'], 
    processing_class= tokenizer, 
    data_collator= collator, 
    compute_metrics= metrics
)

In [109]:
trainer.train()

c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy Score,F1 Score
1,No log,0.700141,0.493100,0.490287
2,No log,0.704891,0.563017,0.548908
3,No log,0.830027,0.587856,0.568401


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.04it/s]
c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.05it/s]
c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  5.16it/s]


TrainOutput(global_step=408, training_loss=0.6393448324764476, metrics={'train_runtime': 478.2671, 'train_samples_per_second': 6.818, 'train_steps_per_second': 0.853, 'total_flos': 62037167410380.0, 'train_loss': 0.6393448324764476, 'epoch': 3.0})

In [110]:
['A'] * 3
# ['A'] + ['A'] + ['A']

['A', 'A', 'A']

In [111]:
# 질문과 답변의 목록들이 입력되었을때 가장 올바른 답변을 찾아주는 함수를 하나 생성 
def score_pairs(question, answers):
    # question : 질문
    # answers : 후보 답변들 
    # 질문은 하나고 답변은 여러개 -> 질문과 답변을 1:1 매칭
    questions = [question] * len(answers)
    tokens = tokenizer(
        questions, 
        answers, 
        truncation = True, 
        max_length = max_len, 
        padding = True, 
        return_tensors = 'pt'
    )
    # token_type_ids 제거 
    tokens.pop('token_type_ids', None)

    # 인코딩 데이터를 딕셔너리 형으로 변환 
    inputs = {
        k : v for k, v in tokens.items()
    }

    model.eval()
    with torch.no_grad():
        logits = model(**inputs)['logits']
        probs = torch.softmax(logits, dim = -1)[:, 1]
        # probs -> 1의 확률 -> 정상적인 답변일 확률
    return probs.tolist()

In [112]:
question = "지방세 납부는 어디서 할 수 있나요?"
answers = [
    '이용하시는 은행 사이트나 앱에서 지방세 납부가 가능합니다', 
    '동물 등록은 거주지 구청에서 처리하셔야 합니다', 
    '출입국 관련 업무는 외교부에서 담당합니다'
]

In [113]:
scores = score_pairs(question, answers)

In [114]:
scores

[0.8732510209083557, 0.22710923850536346, 0.22161811590194702]

In [115]:
# 최대값 -> max()
# 최대값의 위치 -> argmax()
# torch.argmax(torch.tensor(scores))
best_idx = np.argmax(scores)

for idx, (answer, score) in enumerate( zip(answers, scores) ):
    # idx : 위치값
    # answer : 답변
    # scores : 1인 확률
    print(f"{idx} : {answer} -> 적합 확률 : {round(score, 3)}")
print(f"가장 적합한 답변은 {best_idx} : {answers[best_idx]}")

0 : 이용하시는 은행 사이트나 앱에서 지방세 납부가 가능합니다 -> 적합 확률 : 0.873
1 : 동물 등록은 거주지 구청에서 처리하셔야 합니다 -> 적합 확률 : 0.227
2 : 출입국 관련 업무는 외교부에서 담당합니다 -> 적합 확률 : 0.222
가장 적합한 답변은 0 : 이용하시는 은행 사이트나 앱에서 지방세 납부가 가능합니다


In [116]:
import random

In [117]:
# 답변의 목록들 중 확률이 높은 상위 3개 정도의 답변을 출력 
question = random.choice(df3['고객질문'].tolist())
answers = df3['상담사답변'].tolist()

In [118]:
scores = score_pairs(question, answers)

In [119]:
len(scores)

1087

In [120]:

best_idx = np.argmax(scores)
answers[best_idx]

'공공임대는 시나 국가에서 하는 분양이고 민간임대는 말 그대로 민간기업이 분양하는 것을 말합니다.'

In [121]:
question

'용산구에 있는 박물관관련 문의 드립니다.'

In [122]:
# 1일 확률이 높은 상위 3개의 위치
answers_df = pd.DataFrame(zip(answers, scores), columns = ['answer', 'score'])
answers_df.sort_values('score', ascending=False, inplace = True)
answers_df['question'] = question
answers_df.head()

,answer,score,question
429,공공임대는 시나 국가에서 하는 분양이고 민간임대는 말 그대로 민간기업이 분양하는 것...,0.899421,용산구에 있는 박물관관련 문의 드립니다.
270,성동 금호 용담 무지개 성수 청계도서관이 있습니다.,0.893306,용산구에 있는 박물관관련 문의 드립니다.
229,성동 금호 용담 무지개 성수 청계도서관이 있습니다.,0.893306,용산구에 있는 박물관관련 문의 드립니다.
979,성동 금호 용담 무지개 성수 청계도서관 있습니다.,0.887200,용산구에 있는 박물관관련 문의 드립니다.
923,성동 금호 용담 무지개 성수 청계도서관 있습니다.,0.887200,용산구에 있는 박물관관련 문의 드립니다.


In [123]:
# 오름차순 정렬의 인덱스의 값을 확인 
# argsort() : 오름차순 정렬에서 인덱스의 값을 출력 
# score에 - 기호를 사용하여 정렬
# 반대로 뒤집기 [::-1]
# np.flip() : 반대로 뒤집는 함수 
sort_idx = np.argsort(-np.array(scores))
sort_idx

array([429, 270, 229, ...,  40, 708, 395], shape=(1087,))

In [124]:
np.argsort(scores)[::-1]

array([429, 270, 229, ...,  40, 708, 395], shape=(1087,))

In [125]:
np.flip( np.argsort(scores) )

array([429, 270, 229, ...,  40, 708, 395], shape=(1087,))

In [126]:
print(f"질문 : {question}")
for idx in sort_idx[:5]:
    print(f"답변 : {answers[idx]}, 적합 확률 : {round(scores[idx], 4)}")

질문 : 용산구에 있는 박물관관련 문의 드립니다.
답변 : 공공임대는 시나 국가에서 하는 분양이고 민간임대는 말 그대로 민간기업이 분양하는 것을 말합니다., 적합 확률 : 0.8994
답변 : 성동 금호 용담 무지개 성수 청계도서관이 있습니다., 적합 확률 : 0.8933
답변 : 성동 금호 용담 무지개 성수 청계도서관이 있습니다., 적합 확률 : 0.8933
답변 : 성동 금호 용담 무지개 성수 청계도서관 있습니다., 적합 확률 : 0.8872
답변 : 성동 금호 용담 무지개 성수 청계도서관 있습니다., 적합 확률 : 0.8872


In [127]:
# 파인튜닝이 된 모델을 로드 
model2 = BertForSequenceClassification.from_pretrained("./model2")

OSError: Error no file named model.safetensors, or pytorch_model.bin, found in directory ./model2.